# 17 — Sequence Models: RNN, LSTM & GRU

**Learning objective.** Understand recurrent state and train a compact GRU classifier with PyTorch on an inspectable toy corpus.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

## 🧠 Visual engineering mental model

![Causal mindmap](assets/mindmaps/17_sequence_models_rnn_lstm_gru.svg)

Follow the information flow, then ask what the control knob changes before touching code.

## 🎛️ Change map — cause → representation → behavior

| Change | Immediate effect | Downstream consequence |
|---|---|---|
| Increase **sequence length** | hidden state must carry information longer | vanishing memory and compute cost grow |
| Use LSTM/GRU gates | state update becomes selective | longer dependencies are easier to preserve |
| Increase hidden size | state capacity grows | parameters and overfitting risk grow |

**Engineering habit:** make one intervention, state the expected direction of change, then measure it.

## 🔮 Predict before you run

1. Why can a simple RNN forget the first token in a long sequence?
2. What does a gate conceptually decide to keep or overwrite?

### When to use
Useful for understanding sequential state and for some compact/time-series NLP architectures.

### When not / caution
For large modern NLP, transformers often parallelize and model long dependencies better.

### Debugging lens
Track hidden-state behavior, sequence lengths and padding rather than only final loss.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


A recurrent model processes tokens sequentially and maintains a hidden state. LSTM/GRU gating mitigates the difficulty of preserving information over long spans, but recurrence limits parallelism compared with transformers.

In [2]:
import torch
import torch.nn as nn
torch.manual_seed(42)
texts=['good product','excellent value','love it','bad product','terrible value','hate it']*12
labels=torch.tensor([1,1,1,0,0,0]*12)
vocab={'<PAD>':0,'<UNK>':1}
for t in texts:
    for w in t.split(): vocab.setdefault(w,len(vocab))
X=torch.tensor([[vocab.get(w,1) for w in t.split()] for t in texts])
class GRUClassifier(nn.Module):
    def __init__(self):
        super().__init__(); self.emb=nn.Embedding(len(vocab),8); self.gru=nn.GRU(8,12,batch_first=True); self.fc=nn.Linear(12,2)
    def forward(self,x):
        _,h=self.gru(self.emb(x)); return self.fc(h[-1])
model=GRUClassifier(); opt=torch.optim.Adam(model.parameters(),lr=.03); lossfn=nn.CrossEntropyLoss()
for epoch in range(40):
    opt.zero_grad(); logits=model(X); loss=lossfn(logits,labels); loss.backward(); opt.step()
print('final loss:',round(loss.item(),4),'training accuracy:',round((logits.argmax(1)==labels).float().mean().item(),3))

final loss: 0.0 training accuracy: 1.0


In [3]:
print(model)
print('input shape:',tuple(X.shape),'logits shape:',tuple(model(X).shape))

GRUClassifier(
  (emb): Embedding(11, 8)
  (gru): GRU(8, 12, batch_first=True)
  (fc): Linear(in_features=12, out_features=2, bias=True)
)
input shape: (72, 2) logits shape: (72, 2)


---
    ## Production takeaways
    - Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
    - Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
    - Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Explain recurrent hidden state
- Connect tensor shapes to sequence classification